# M2 — G1 đi bộ trên mặt phẳng 
Train M2 (MLP 896-448-128, input `o_t`) bằng MuJoCo Playground + Brax PPO trên Kaggle (2×T4).

- Môi trường: địa hình `flat` + cấu hình của Minh (`configs/terrain.yaml`, `UnifiedG1TerrainEnv`) — **không sửa**.
- Phần thêm mới nằm trong `flat_walking/` (xem `flat_walking/readme_report.md`).
- Chạy lần lượt từ Cell 1. Cell 5 là kiểm tra nhanh, nên chạy trước Cell 6 (train).

In [3]:
# Cell 1: Kiểm tra GPU
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB



In [4]:
# Cell 2: Clone / cập nhật repo (nhánh M2)
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")
if not PROJECT_DIR.exists():
    subprocess.check_call(["git", "clone", "--branch", "M2",
                           "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git", str(PROJECT_DIR)])
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %an %s"], capture_output=True, text=True).stdout)

Cloning into '/kaggle/working/Bipedal-Locomotion'...


Working dir: /kaggle/working/Bipedal-Locomotion
9c3bac4 OanhKim-tmd feat(M2): result backup/download tools, reports, and 100M step budget



In [5]:
# Cell 3: Cài dependencies
import sys, subprocess

pkgs = ["brax", "flax", "optax", "orbax-checkpoint", "ml_collections", "pyyaml",
        "lxml", "etils", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

try:
    import mujoco_playground
    print("[OK] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"])
    print("[OK] mujoco_playground installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 119.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 130.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 118.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 kB 27.0 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully uninstalled PyYAML-6.0.3
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attemp

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


  Cloning https://github.com/google-deepmind/mujoco_playground.git (to revision v0.2.0) to /tmp/pip-req-build-eza2ltxr


  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/mujoco_playground.git /tmp/pip-req-build-eza2ltxr
  Running command git checkout -q 124a73fa3303f75a62f8fe04d329b829ed0ebdfb


  Resolved https://github.com/google-deepmind/mujoco_playground.git to commit 124a73fa3303f75a62f8fe04d329b829ed0ebdfb
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for playground: filename=playground-0.2.0-py3-none-any.whl size=15667966 sha256=bca134b202c67df8c3fccfde77d77a6ce399a94ff4010c596be4f46fb1accf37
  Stored in directory: /tmp/pip-ephem-wheel-cache-micmpr2z/wheels/98/90/d2/b6ca0a9f640e11b462835f83b75a05eaa86982773376370d44
Successfully built playground
[OK] mujoco_playground installed


In [6]:
# Cell 4: Kiểm tra JAX / GPU
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
import jax
print(f"JAX {jax.__version__}:", jax.devices())

JAX 0.11.1: [CudaDevice(id=0), CudaDevice(id=1)]


In [7]:
# Cell 5: Kiểm tra nhanh môi trường
import subprocess, sys
r = subprocess.run([sys.executable, "flat_walking/train_m2_flat.py", "--smoke"],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1500:] if r.returncode else "")
assert r.returncode == 0, "smoke test lỗi — đừng chạy Cell 6"


/usr/bin/python3: can't open file '/kaggle/working/Bipedal-Locomotion/flat_walking/train_m2_flat.py': [Errno 2] No such file or directory



AssertionError: smoke test lỗi — đừng chạy Cell 6

In [ ]:
# Cell 6: TRAIN M2 (100M bước, ~vài chục phút -> vài giờ tuỳ GPU)
import subprocess, sys

NUM_TIMESTEPS = None     # None = lấy từ flat_walking/m2_flat.yaml (100M). Thử nhanh: 2_000_000
cmd = [sys.executable, "-u", "flat_walking/train_m2_flat.py"]
if NUM_TIMESTEPS:
    cmd += ["--num-timesteps", str(NUM_TIMESTEPS)]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "train lỗi"

In [ ]:
# Cell 7: Nén kết quả để tải về (results/flat -> /kaggle/working/m2_flat_results.zip)
import shutil
print(shutil.make_archive("/kaggle/working/m2_flat_results", "zip", "results/flat"))